# 01 · Bronze Layer Data Exploration
**Simulates a Microsoft Fabric Notebook** — run `python lakehouse.py` first to populate the data.

In [ ]:
import sys, os
os.chdir('..')  # run from repo root
import duckdb, pandas as pd
con = duckdb.connect('data/retailco_lakehouse.duckdb')
print('Connected to DuckDB (SQL Analytics Endpoint simulation)')

## Schema inspection
In Fabric: right-click a table in the Lakehouse explorer → 'Schema'

In [ ]:
con.execute('SHOW TABLES').df()

In [ ]:
con.execute('DESCRIBE silver.orders').df()

## Row counts & freshness

In [ ]:
for t in ['silver.orders','gold.fct_revenue_daily','gold.dim_customer_ltv']:
    n=con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:<35} {n:>10,} rows')

## Sample rows

In [ ]:
con.execute('SELECT * FROM silver.orders LIMIT 5').df()

## Channel distribution

In [ ]:
con.execute('''
    SELECT channel, COUNT(*) AS orders,
           ROUND(SUM(total_amount)/1e6,2) AS rev_M
    FROM silver.orders GROUP BY channel ORDER BY rev_M DESC
''').df()

## Loyalty tier breakdown

In [ ]:
con.execute('''
    SELECT loyalty_tier, COUNT(DISTINCT customer_id) AS customers,
           ROUND(AVG(total_amount),2) AS avg_order
    FROM silver.orders GROUP BY loyalty_tier ORDER BY avg_order DESC
''').df()